# Pairwise similarity distribution: Find sentences that are similar to eachother

This notebook uses the cosine similarity scores to find sentences that appear to be the most similar to eachother.
- Distribution of all similarity scores.
- Plot sentences of two papers in a cluster map.
---
- Creates a data frame with sentence pairs.
- Distribution of highest similarity scores.
- Check context of sentences with high 'highest similarity score'.
- Check context of sentences with low 'highest similarity score'.
---

- Explore paper pairs

### Settings

In [1]:
# Settings
experiment_name = "free_1000_251013_pest_PD"

## Initialisation

### Imports

In [2]:
# import
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.diagnostic import normal_ad

import torch

### Functions

In [3]:
def df_describe_not_norm(series):
    """Return dataframe with describing values for a non-normal distribution."""
    # create dataframe with median, min, and max
    new_df = pd.DataFrame.from_dict({
            "count": series.shape[0],
            "median": series.median(),
            "min": series.min(),
            "max": series.max()
    }, orient= 'index').T

    # add value range
    new_df['range'] = new_df['max'] - new_df['min']

    return new_df.T

def similarity_to_pairs(corpus: pd.DataFrame, matrix: torch.Tensor) -> pd.DataFrame:
    """Create a pd.DataFrame where the sentence in rows are paired with the sentence that has the highest similarity score (from `matrix`) and the paper_name and sentence_text are added from `corpus` for context."""
    # Substract 2 from the cosine similarity score that matched itself, so that it turns the value to -1.
    matrix_substracted = np.subtract(matrix, np.identity(matrix.shape[0]) *2)

    # Retrieve index of sentence that is most similar to sentence at array position.
    # Example: [1, 0, 5, ...] -> 0st sentence matches best to 1nd sentence, 1nd " 0st, 3rd " 5th.
    sent_pairs_text = matrix_substracted.numpy().argmax(axis=0)
    # Retrieve the similarity score for the pairs
    sent_pairs_score = np.array([pd.DataFrame(matrix).loc[i_r, i_c] for i_r, i_c in enumerate(sent_pairs_text)])

    # Gather info into one data frame
    df_sent_pairs = pd.DataFrame(
        {
            'i_c': sent_pairs_text, # sentence_id of cos_sim with highest score.
            'cos_sim': sent_pairs_score # Highest cos_sim score for sentence.
        }
    )
    # Add paper_name (= pmid) by row id (i_r)
    df_sent_pairs = pd.merge(df_sent_pairs, corpus[['paper_name','sentence_text']], left_index=True, right_index=True)
    # Add paper_name (= pmid) by column id (i_c)
    df_sent_pairs = pd.merge(df_sent_pairs, corpus[['paper_name','sentence_text']], left_on='i_c', right_index=True)
    # Rename the just added columns
    df_sent_pairs = df_sent_pairs.rename(
        columns= {
            'paper_name_x': 'i_r_pmid',
            'paper_name_y': 'i_c_pmid',
            'sentence_text_x': 'i_r_text',
            'sentence_text_y': 'i_c_text',
        }
    )

    # Add name to index
    df_sent_pairs.index.name = 'i_r'
    
    return df_sent_pairs

### Load vectors and corpus

In [4]:
# Filenames
similarity_file = f"similarity_{experiment_name}.pickle"
raw_corpus_file = f"corpus_{experiment_name}.csv"

In [5]:
# Load raw corpus
df_corpus = pd.read_csv(f'../../data/corpus/{raw_corpus_file}')

In [6]:
# Load cosine similarity scores
with open(f"../../data/vectors/{similarity_file}", 'rb') as handle:
    similarities = pickle.load(handle)

In [7]:
# Check if similarity score is loaded correctly
if np.sum(np.array(similarities.diagonal())) == similarities.shape[0]:
    print("Similarity scores appear to be correctly loaded.")

else:
    raise ValueError(f"Something went wrong with loading the similarity scores? The sum of the diagonal appears not to be the same as the total number of rows. Sum = {np.sum(np.array(similarities.diagonal()))}, number of rows = {embeddings.shape[0]} (Could be a float point error?)")

Similarity scores appear to be correctly loaded.


/tmp/ipykernel_1204528/3763688713.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  if np.sum(np.array(similarities.diagonal())) == similarities.shape[0]:


In [8]:
# sample
similarities[:5, :5]

tensor([[1.0000, 0.1074, 0.2322, 0.5048, 0.3413],
        [0.1074, 1.0000, 0.4907, 0.3195, 0.5344],
        [0.2322, 0.4907, 1.0000, 0.3907, 0.4358],
        [0.5048, 0.3195, 0.3907, 1.0000, 0.3484],
        [0.3413, 0.5344, 0.4358, 0.3484, 1.0000]])

Similarity scores are correctly loaded if row and column indices with the same number == `1`

## Visualise similarity

### Total similarity distribution

To get a feeling for the data, we explore the similarity score distribution.

In [ ]:
# Get a flat list of cosine similarity scores 
# (NOTE: Takes long: ~9 minutes for 30484 x 30484 sentences)
# https://stackoverflow.com/questions/8905501/extract-upper-or-lower-triangular-part-of-a-numpy-matrix
sim_flat = similarities[np.triu_indices(similarities.shape[0], k=1)]

In [ ]:
# Plot distribution
plt.hist(
    sim_flat,
    bins= np.arange(-0.2,1.2,0.05)
)

# Add median line
med = sim_flat.median()
plt.vlines(med, 0, 71000000, color='orange', label= f'median ({med:.3f})')

# Get described information
describe_info = df_describe_not_norm(pd.Series(sim_flat)).T
textstr = "\n".join([
    f"Count: {describe_info["count"][0]:.0f}",
    f"Median: {describe_info["median"][0]:.3f}",
    f"Minimum: {describe_info["min"][0]:.3f}",
    f"Maximum: {describe_info["max"][0]:.3f}",
    f"Range: {describe_info["range"][0]:.3f}",
])
props = dict(boxstyle='round', 
             facecolor='white', 
             alpha=0.5)
plt.text(
    0.8,
    80000000,
    textstr,
    bbox = props, 
    verticalalignment='top'
)

plt.xlabel('Cosine similarity score')
plt.ylabel("Sentence frequency")

plt.title('Distribution of all cosine similarity score')
plt.show()

In [ ]:
# Test if normal distributed
print(f"p-value for Anderson-Darling test: {normal_ad(sim_flat)[1]}")

It appears somewhat normal distributed with a heavy tail to the right.

In [ ]:
# Describe distribution
df_describe_not_norm(pd.Series(sim_flat)).T

Note that in the similarity scores there are floating point errors resulting in the possibility of scores larger than 1.

### Plot sentences of two papers in a cluster map

Only a subsection is plotted in the cluster map, since plotting all 30484 sentences would be too much.
The first 279 sentences come from the following two papers (PMID): 17900545, 32375810.
```
PMID: first sentence - last sentence
17900545: 0 - 157
32375810: 158 - 279
```

In [ ]:
# plot clustermap
subset = 279
sns.clustermap(similarities[:subset, :subset], vmin=0, vmax=1)
plt.title(f"Sentence similarty (cosine) on a subset ({subset}) of corpus")
plt.show()

I expect a paper to contain a lot of sentences that are the same. Therefore I expected to see two large clusters since the sentences come from two different papers.

## Explore sentence pairs

### Create a data frame with sentence pairs

In [ ]:
# Create a pd.DataFrame where the sentence in rows are paired with the sentence that has the highest similarity score and add context data.
df_sent_pairs = similarity_to_pairs(df_corpus, similarities)
df_sent_pairs.head()

### Distribution of highest similarity scores

In [ ]:
# Distribution of highest cosine similarity score for a sentence
df_sent_pairs['cos_sim'].hist(bins= np.arange(0,1.2,0.05))

plt.xlabel('Cosine similarity score')
plt.ylabel("Sentence frequency")

# Add median line
plt.vlines(df_sent_pairs['cos_sim'].median(), 0, 6350, color='orange', label= f'median ({df_sent_pairs['cos_sim'].median():.3f})')

# Get described information
describe_info = df_describe_not_norm(df_sent_pairs['cos_sim']).T
textstr = "\n".join([
    f"Count: {describe_info["count"][0]:.0f}",
    f"Median: {describe_info["median"][0]:.3f}",
    f"Minimum: {describe_info["min"][0]:.3f}",
    f"Maximum: {describe_info["max"][0]:.3f}",
    f"Range: {describe_info["range"][0]:.3f}",
])
props = dict(boxstyle='round', 
             facecolor='white', 
             alpha=0.5)
plt.text(
    0,
    6000,
    textstr,
    bbox = props, 
    verticalalignment='top'
)


plt.title('Distribution of highest cosine similarity score per sentence')
plt.grid(False)

plt.show()

In [ ]:
# Test if normal distributed
print(f"p-value for Anderson-Darling test: {normal_ad(df_sent_pairs['cos_sim'].to_numpy())[1]}")

It appears somewhat normal distributed with a heavy tail to the left.

In [ ]:
# Describe distribution
df_describe_not_norm(df_sent_pairs['cos_sim']).T

It shows that if we place a threshold around the median (= ~0.75), it will exclude ~50% of the sentences.

Note that there are a few sentences that are 1=<, suggesting sentences to be duplicates. And note again that in the similarity scores there are floating point errors resulting in the possibility of scores larger than 1.

See next section for further break down.

### Investigate sentence pairs

In [ ]:
# (see print)
if df_sent_pairs.shape[0] == similarities.shape[0]:
    print(f"Note that `df_sent_pairs` contains all sentence pairs, incl. duplicates.")

else:
    print("Duplicate pairs have been removed from `df_sent_pairs`.")

##### 'best' pairs

Note that there are a few sentences that are 1=<, suggesting sentences to be duplicates.

In [ ]:
# How many sentences have cos_sim of 1 or higher?
# Number of sentences that appear to be exactly the same:
print(f"Number of sentences that appear to be exactly the same: {df_sent_pairs[df_sent_pairs['cos_sim'] >= 1].shape[0]}")

In [ ]:
# Display more characters in printed data frames
pd.set_option('display.max_colwidth', 110)

In [ ]:
# Show samples
df_sent_pairs[df_sent_pairs['cos_sim'] >= 1].sample(10)

Possibly this is explained by papers using the same header multiple times in different sections. Duplicates also contain standard disclaimers (e.g. 'The authors declare no competing interests.', etc.). Though there are also longer sentences that appear quiet specific.

It appears that PMID 37292848 and PMID 38052871 are near identical papers, were the first is not peer-reviewed, but the second is.

In [ ]:
# Look at sentence pairs with the highest similarity
df_sent_pairs.sort_values('cos_sim', ascending= False).head(20)

(Interesting to report!) The top 20 includes three different papers from one and the same authors. pmid = [11826108, 12598611, 12867501].

Then there are sentences that are 'standard disclaimers'. (i.e. 'The authors declare no conflict of interest.', 'Written informed consent was obtained from...', etc.)

Then there are sentences that are used as headers. Like for sentence_id 16262 and 15773.


##### 'worst' pairs

Check if sentences with a low similarity score do not seem similar. We expect these sentences are considered to be 'unique' in the corpus, given that they did not manage to get a high match with another sentence in the corpus.

In [ ]:
# Sentences with the lowest 'highest similarity match'
df_sent_pairs.sort_values('cos_sim', ascending= True).head(10)

In [ ]:
# Get full sentences
for row in df_sent_pairs.sort_values('cos_sim', ascending= True).head(5).values:
    print(f'cosine similarity score: {row[1]:.3f}\nSentence 1: "{row[3]}"\nSentence 2: "{row[5]}"')
    print("~~~")

To me these sentences already appear seem somewhat similar. (the first is about 'Transcranial B-mode sonography monitors', the second about knee/joint injury and swellings)

### Sample Time!

Sample through the matches and read the full text.

In [ ]:
# Sample pairs from the df
i_sample = df_sent_pairs.sample(10).index.to_numpy()
df_sent_pairs.loc[i_sample]

In [ ]:
# Print the metrics and sentences
for i in i_sample:
    print(f"\ni_c: {df_sent_pairs.loc[i, 'i_c']} | i_r: {df_sent_pairs.loc[i].name} | Cosine similairty score: {df_sent_pairs.loc[i, 'cos_sim']}")
    print(f"Sentence row:    {df_sent_pairs.loc[i, 'i_r_text']}")
    print(f"Sentence column: {df_sent_pairs.loc[i, 'i_c_text']}")
    print("---")

~END~